In [1]:
import numpy as np
import pandas as pd
import altair as alt
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def logistic_trajectory(x0, r, n=50):
    values = [x0]
    for _ in range(n):
        values.append(r * values[-1] * (1 - values[-1]))
    return values


def show_two_initials():
    r_slider = widgets.FloatSlider(
        value=2.5, min=2.5, max=3.99, step=0.01,
        description="r:", continuous_update=False, readout_format=".2f")
    epsilon_slider = widgets.FloatLogSlider(
        value=0.001, base=10, min=-5, max=-1, step=0.1,
        description="ε:", continuous_update=False, readout_format=".6f")
    output = widgets.Output()

    def draw(change=None):
        r, epsilon = r_slider.value, epsilon_slider.value
        starts = [0.10, 0.10 + epsilon]
        labels = [f"x₀ = {x0:.6f}" for x0 in starts]
        colors = alt.Scale(domain=labels, range=["steelblue", "orange"])
        color = alt.Color("initial:N", title="Initial condition", scale=colors,
                          legend=alt.Legend(orient="bottom"))

        # Both initial conditions use exactly the same rule.
        x = np.linspace(0, 1, 500)
        rule_data = pd.concat([
            pd.DataFrame({"x": x, "next": r*x*(1-x), "initial": label})
            for label in labels
        ], ignore_index=True)
        rule_base = alt.Chart(rule_data).encode(
            x=alt.X("x:Q", title="xₜ", scale=alt.Scale(domain=[0, 1])),
            y=alt.Y("next:Q", title="xₜ₊₁", scale=alt.Scale(domain=[0, 1])),
            color=color)
        rule = alt.layer(
            rule_base.transform_filter(alt.datum.initial == labels[0])
                     .mark_line(strokeWidth=6),
            rule_base.transform_filter(alt.datum.initial == labels[1])
                     .mark_line(strokeWidth=2.5, strokeDash=[2, 5]),
        ).properties(width=390, height=330, title="Same rule: the curves overlap")

        trajectories = pd.concat([
            pd.DataFrame({"t": np.arange(51),
                          "x": logistic_trajectory(x0, r), "initial": label})
            for x0, label in zip(starts, labels)
        ], ignore_index=True)
        time_series = (
            alt.Chart(trajectories).mark_line(point=True, strokeWidth=2)
            .encode(
                x=alt.X("t:Q", title="Iteration", scale=alt.Scale(domain=[0, 50])),
                y=alt.Y("x:Q", title="xₜ", scale=alt.Scale(domain=[0, 1])),
                color=color,
                strokeDash=alt.StrokeDash(
                    "initial:N", title="Initial condition",
                    scale=alt.Scale(domain=labels, range=[[1, 0], [6, 3]]),
                    legend=None),
                tooltip=[alt.Tooltip("initial:N", title="Initial condition"),
                         alt.Tooltip("t:Q", title="Iteration"),
                         alt.Tooltip("x:Q", title="xₜ", format=".8f")])
            .properties(width=390, height=330, title="Two nearby starting points")
        )
        chart = alt.hconcat(rule.interactive(), time_series.interactive()).properties(
            title=f"Same rule | r = {r:.2f} | ε = {epsilon:.6f}"
        ).resolve_scale(x="independent", y="independent", color="shared")
        with output:
            clear_output(wait=True)
            display(chart)

    for control in (r_slider, epsilon_slider):
        control.observe(draw, names="value")
    display(widgets.VBox([
        widgets.HBox([r_slider, epsilon_slider]), output]))
    draw()


show_two_initials()
